# Clase 8 — Datos geoespaciales y pipeline integrado

## Pregunta central

> **¿Cómo combinamos píxeles, coordenadas y límites de parcelas para obtener información útil?**

## Idea principal

Un dato geoespacial combina una medición con una ubicación; para integrarlo correctamente necesitamos conocer su formato, resolución y CRS.

## Objetivos de aprendizaje

Al finalizar la clase deberías poder:

- Distinguir datos ráster y vectoriales.
- Interpretar bandas, resolución, extensión, transform y CRS de un GeoTIFF.
- Leer un GeoJSON y comprobar que comparte el CRS del ráster.
- Calcular NDVI y NDWI sin tratarlos como verdades absolutas.
- Relacionar clasificación, detección y segmentación con datos georreferenciados.
- Crear una máscara de parcela y exportar un GeoTIFF derivado.

## Recorrido de la clase

| Paso | Tema |
|---:|---|
| 1 | Qué hace geoespacial a un dato |
| 2 | Ráster, vector y operaciones entre capas |
| 3 | CRS, proyecciones y resolución |
| 4 | Sensores, bandas y productos de imagen |
| 5 | Dónde aparece la IA |
| 6 | Práctica: abrir GeoTIFF y GeoJSON |
| 7 | Índices, máscaras y superposición |
| 8 | Exportar un resultado |
| 9 | Actividad y pipeline de producción |

## Cómo trabajar con este notebook

1. Ejecutá las celdas en el orden propuesto.
2. Antes de modificar código, observá y describí el resultado.
3. Cambiá solamente las variables marcadas con `TODO`.
4. No es necesario implementar algoritmos desde cero.
5. Si aparece un término nuevo, buscá primero su definición en el glosario de la clase.

**Conexión con el programa:** las clases de imágenes satelitales, segmentación y pipelines agro del Track Imagen

---
## 1. ¿Qué hace geoespacial a un dato?

Un dato es **geoespacial** cuando sabemos qué representa y dónde se
encuentra sobre la Tierra. La ubicación permite formular preguntas
que no podríamos responder con una tabla o imagen aislada:

- ¿qué lotes intersectan una zona inundada?;
- ¿cuántos metros de una ruta necesitan reparación?;
- ¿cómo cambió la vegetación de una parcela entre dos fechas?;
- ¿dónde detectó un modelo edificios o árboles?;
- ¿qué valor promedio tiene una imagen dentro de cada municipio?

El dato visible y su referencia espacial forman una unidad:

```text
valores o geometrías
        +
sistema de coordenadas
        +
ubicación, escala y fecha
        =
dato geoespacial interpretable
```

Una matriz de píxeles sin ubicación sigue siendo una imagen. Un
GeoTIFF agrega la información necesaria para ubicar esa matriz en un
mapa. Del mismo modo, una lista de pares numéricos solo se convierte
en una geometría útil cuando conocemos qué significan sus
coordenadas.

### Dos modelos de datos complementarios

| Modelo | Unidad básica | Sirve para | Formatos frecuentes |
|---|---|---|---|
| **Ráster** | Celda o píxel de una grilla | Fenómenos continuos: imágenes, elevación, temperatura, índices | GeoTIFF, COG |
| **Vectorial** | Punto, línea o polígono | Objetos o límites: pozos, rutas, parcelas, provincias | GeoJSON, GeoPackage |

Un **COG** es un GeoTIFF organizado para leer partes remotas del
archivo sin descargarlo completo. Un **GeoPackage** es un archivo de
base de datos que puede guardar varias capas vectoriales o ráster y
su referencia espacial.

Un píxel de una fotografía común tiene color. Un píxel geoespacial
también tiene una **posición sobre la Tierra** y puede contener varias
mediciones: azul, verde, rojo, infrarrojo, temperatura o altura.

Un vector separa **geometría** y **atributos**. Por ejemplo, un
polígono describe el límite de una parcela y su fila puede guardar
`id`, cultivo, productor o fecha de siembra. No contiene la imagen:
indica sobre qué parte de la imagen queremos trabajar.

### Operaciones frecuentes entre capas

| Operación | Pregunta que responde |
|---|---|
| Superposición | ¿Las capas ocupan el mismo lugar? |
| Recorte | ¿Qué parte del ráster cae dentro de esta región? |
| Rasterización | ¿Qué píxeles corresponden a un polígono? |
| Vectorización | ¿Qué geometrías forman los píxeles seleccionados? |
| Estadística zonal | ¿Cuál es el promedio, mínimo o máximo dentro de cada polígono? |

Esta clase recorre una combinación muy común:

```text
imagen multibanda + polígono de parcela
         ↓
 alineación y máscara espacial
         ↓
 índices por píxel
         ↓
 estadísticas y producto exportable
```

### Distribución sugerida de las dos horas

| Tramo | Tiempo |
|---|---:|
| Modelos de datos, CRS y sensores | 35 min |
| Inspección y visualización de capas | 25 min |
| Índices, máscara y exportación | 35 min |
| Actividad, discusión y cierre | 25 min |

## Glosario mínimo

| Término | Explicación breve |
|---|---|
| GeoTIFF | Imagen TIFF con ubicación, resolución y CRS |
| GeoJSON | Documento JSON con geometrías y atributos |
| Píxel o celda | Posición de la grilla que almacena uno o más valores |
| Banda | Una medición por píxel, por ejemplo rojo o infrarrojo |
| CRS | Sistema que da significado a las coordenadas |
| Proyección | Forma de representar la Tierra curva sobre un plano |
| Código EPSG | Identificador conocido de un CRS, como `EPSG:4326` |
| Resolución espacial | Tamaño del terreno representado por un píxel |
| Extensión | Rectángulo de coordenadas cubierto por el dato |
| Transform | Regla que convierte fila/columna en coordenadas |
| Máscara | Matriz que selecciona qué píxeles usar |
| Índice espectral | Combinación matemática de bandas |
| Reflectancia | Proporción de energía que una superficie refleja |
| Nodata | Valor que representa ausencia de información |
| GIS | Sistema de Información Geográfica para almacenar, consultar, transformar y combinar capas |
| COG | GeoTIFF optimizado para acceso parcial, incluso por red |
| GeoPackage | Archivo de base de datos espacial que puede contener varias capas |
| Revisita | Tiempo entre observaciones sucesivas de un satélite sobre una zona |
| Tile o tesela | Bloque rectangular usado para procesar una imagen grande por partes |

> Dos capas pueden verse parecidas y aun así no superponerse si sus
> coordenadas pertenecen a CRS diferentes.

No es necesario memorizar todos los términos. Durante la práctica
veremos que cada uno corresponde a una propiedad concreta del
archivo o a una operación observable.

---
## 2. CRS, proyecciones, capas y resolución

### El CRS es el contrato de las coordenadas

Las coordenadas `-26.8, -65.2` parecen grados de latitud/longitud.
Las coordenadas `350000, 7340000` parecen metros en una proyección.
Sin el **CRS** son solo dos números: no sabemos origen, unidad,
orientación ni zona válida.

Un CRS reúne decisiones sobre cómo describir posiciones terrestres.
Para esta introducción alcanza distinguir dos familias:

| Familia | Coordenadas habituales | Ventaja | Ejemplo |
|---|---|---|---|
| Geográfico | Longitud y latitud en grados | Intercambio global | WGS 84, `EPSG:4326` |
| Proyectado | X e Y, normalmente en metros | Medir y analizar en un plano | UTM 20S, `EPSG:32720` |

La Tierra es curva. Una **proyección** la representa sobre un plano y
necesariamente distorsiona algo: área, distancia, forma o dirección.
Por eso no existe un CRS proyectado perfecto para todo el planeta.
Elegimos uno adecuado para la región y la medición que necesitamos.

En esta práctica usaremos `EPSG:32720`:

- UTM, zona 20 sur;
- coordenadas expresadas en metros;
- apropiado para medir distancias y superficies dentro de esa zona.

### Asignar un CRS no es reproyectar

Esta distinción evita uno de los errores más peligrosos:

```text
asignar CRS
"estos números ya estaban expresados en este sistema"

reproyectar
"transformar los números desde un sistema conocido hacia otro"
```

Cambiar solamente la etiqueta del CRS no mueve ni convierte las
coordenadas. Si una capa está en grados y otra en metros, primero
debemos conocer correctamente ambos CRS y luego **reproyectar**.

### Resolución, escala y exactitud no son sinónimos

La escena tiene una resolución de **10 metros por píxel**. Un píxel representa
aproximadamente `10 m × 10 m = 100 m²`. Esto no significa que podamos
reconocer objetos de 10 m con precisión perfecta: sensores, atmósfera,
remuestreo y mezcla de coberturas también influyen.

- **Resolución:** tamaño del terreno cubierto por una celda.
- **Exactitud posicional:** qué tan cerca está el dato de su ubicación real.
- **Escala de análisis:** tamaño del fenómeno que queremos estudiar.

Un árbol pequeño puede ocupar una fracción de un píxel de 10 m. Ese
píxel mezcla árbol, suelo y sombra. Aumentar artificialmente el tamaño
de la imagen no crea detalle nuevo.

### Grilla, extensión y alineación

Dos rásteres pueden compartir CRS y resolución, pero tener sus celdas
desplazadas. Para operar píxel a píxel también deben compartir:

- ancho y alto compatibles;
- transform y origen de la grilla;
- extensión espacial;
- criterio de `nodata`;
- fecha o período comparables.

### Regla de integración

Antes de cruzar capas, comprobar:

1. que ambas tengan CRS;
2. que los CRS sean iguales o reproyectar una capa;
3. que sus extensiones se intersecten;
4. que las grillas estén alineadas si habrá operaciones píxel a píxel;
5. que la resolución y fecha sean adecuadas para la pregunta.

```text
capa A: CRS + extensión + resolución + transform + fecha
                          │
                          ▼
                     validación
                          ▲
                          │
capa B: CRS + extensión + resolución + transform + fecha
```

> **Nota de interoperabilidad:** GeoJSON moderno (RFC 7946) asume
> coordenadas WGS 84. El asset didáctico incluye el miembro `crs`
> heredado para poder practicar dos capas en UTM. Para intercambiar
> datos con clientes web conviene reproyectar el GeoJSON a
> `EPSG:4326`; para conservar otro CRS, GeoPackage suele ser una
> alternativa más segura.

---
## 3. ¿De dónde vienen las imágenes?

Una imagen geoespacial no aparece lista para analizar. Un sensor mide
energía, el proveedor procesa esas mediciones y finalmente recibimos
una o más bandas con metadatos.

| Fuente | Resolución y cobertura | Fortalezas | Limitaciones típicas |
|---|---|---|---|
| Dron | Centímetros; predios o zonas pequeñas | Mucho detalle, captura a demanda | Plan de vuelo, volumen, regulación y posprocesamiento |
| Satélite | Metros a kilómetros; regiones extensas | Cobertura amplia y series históricas | Nubes, revisita y menor detalle |
| Avión | Decímetros o metros; cobertura intermedia | Sensores especializados | Costo y disponibilidad |

Un dron obtiene muchas fotos parcialmente superpuestas. Un
**ortomosaico** combina y corrige esas imágenes para crear una capa
continua con geometría cartográfica. El flujo simplificado es:

```text
fotos superpuestas + posición/orientación
                  ↓
        correspondencias entre fotos
                  ↓
      modelo del terreno y correcciones
                  ↓
             ortomosaico
```

Una imagen pegada visualmente no es necesariamente un ortomosaico:
la corrección geométrica es lo que permite medir y superponer capas.

Una escena puede ser:

- **RGB:** rojo, verde y azul; similar a una fotografía;
- **multibanda:** agrega mediciones no visibles, como infrarrojo
  cercano (NIR);
- **producto derivado:** índice, clasificación o máscara producida
  a partir de otras bandas.

### Bandas y firma espectral

Cada material interactúa de manera diferente con distintas regiones
del espectro. Una planta sana suele absorber bastante luz roja para
fotosíntesis y reflejar más infrarrojo cercano. El agua, suelo,
hormigón y vegetación producen combinaciones diferentes.

```text
             un mismo píxel
                  │
    ┌─────────────┼─────────────┐
    ▼             ▼             ▼
 rojo=0.09    verde=0.23     NIR=0.64
    └─────────────┬─────────────┘
                  ▼
         vector de características
```

Una banda no es una clase. El valor `NIR=0.64` no significa por sí
solo “vegetación”: su interpretación depende de las demás bandas,
calibración, fecha y contexto.

### De medición cruda a producto analizable

| Etapa | Pregunta de control |
|---|---|
| Adquisición | ¿Qué sensor, fecha y condiciones se usaron? |
| Corrección | ¿Los valores y la geometría son comparables? |
| Nubes y nodata | ¿Qué píxeles deben excluirse? |
| Alineación | ¿Las bandas y fechas comparten grilla? |
| Análisis | ¿Qué índice, regla o modelo responde la pregunta? |
| Validación | ¿Hay observaciones de campo o referencias confiables? |

En proyectos grandes, plataformas como **Google Earth Engine**
permiten consultar catálogos y procesar colecciones en la nube.
**Sentinel Hub** ofrece APIs para buscar y procesar imágenes de
distintas misiones.

Esas plataformas no eliminan la necesidad de comprender bandas, CRS
o calidad; cambian dónde viven los datos y dónde se ejecuta el
procesamiento. Aquí usamos archivos locales para que cada paso quede
visible y no requiera cuentas ni claves.

---
## 4. ¿Dónde aparece la inteligencia artificial?

Un **GIS** (*Geographic Information System* o Sistema de Información
Geográfica) reúne herramientas para trabajar con datos ubicados en
la Tierra. GIS e IA no son lo mismo:

- **GIS** aporta representación espacial, transformaciones, consultas
  y combinación de capas;
- **Machine Learning** aprende relaciones a partir de ejemplos;
- **Deep Learning** puede aprender características directamente de
  imágenes, audio o texto;
- un producto útil suele integrar las tres cosas con reglas de
  negocio y validaciones.

### La unidad de predicción cambia el problema

| Tarea | Entrada posible | Salida | Ejemplo |
|---|---|---|---|
| Clasificación tabular | Estadísticas por parcela | Una clase por fila | “riesgo alto/medio/bajo” |
| Clasificación de imagen | Recorte o tile | Una clase por imagen | “contiene cultivo” |
| Detección | Imagen | Cajas + clases | Localizar árboles o vehículos |
| Segmentación | Imagen | Clase por píxel | Delimitar agua o cultivo |
| Regresión | Features o imagen | Valor continuo | Estimar rendimiento |
| Detección de cambio | Imágenes de dos fechas | Mapa de cambio | Área desmontada |

```text
datos geoespaciales
       │
       ├── features por parcela ──→ modelo tabular
       │
       ├── recortes de imagen ────→ clasificador/detector
       │
       └── píxeles multibanda ────→ segmentador
                                        │
                                        ▼
                                  máscara ráster
                                        │
                                        ▼
                          cruce con parcelas y métricas
```

En la práctica de hoy no entrenaremos un modelo. Una regla de NDVI
producirá una máscara con la misma estructura que podría entregar un
segmentador. Esto permite aprender la integración antes de agregar
complejidad de entrenamiento.

### Riesgo particular: fuga espacial y temporal

Píxeles vecinos y capturas cercanas suelen parecerse mucho. Si
dividimos aleatoriamente píxeles de una misma parcela entre train y
test, el modelo puede parecer excelente porque ve casi el mismo lugar
en ambos conjuntos.

En proyectos reales se separa por región, parcela o período cuando
esa separación representa mejor el uso futuro. Una métrica alta no
compensa un diseño de evaluación contaminado.

---
## 5. Práctica: abrir un GeoTIFF y un GeoJSON

Los assets son pequeños, sintéticos y reproducibles:

- cuatro bandas: azul, verde, rojo y NIR;
- una escena de 96 × 96 píxeles;
- una parcela vectorial dentro de la escena;
- CRS `EPSG:32720` en ambos archivos.

La escena simula tres coberturas fáciles de reconocer: vegetación,
suelo y agua. No representa una ubicación ni un cultivo reales y no
se usará para afirmar calidad productiva.

### Herramientas que usaremos

| Biblioteca | Responsabilidad en el notebook |
|---|---|
| NumPy | Arreglos y operaciones por píxel |
| Rasterio | Lectura, metadatos y escritura de GeoTIFF |
| GeoPandas | Lectura y operaciones con el GeoJSON |
| Matplotlib | Visualización y superposición de capas |

`FAST_MODE` se mantiene por consistencia con el curso. En esta clase
todo el dataset ya es pequeño, por lo que no hace falta muestrearlo.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from IPython.display import display
from matplotlib.colors import ListedColormap
from rasterio.features import geometry_mask
from rasterio.plot import plotting_extent

FAST_MODE = True
SEED = 42

PROJECT_ROOT = Path.cwd()
ASSET_DIR = PROJECT_ROOT / "assets" / "geospatial"
if not ASSET_DIR.exists():
    # Permite ejecutar el notebook desde un subdirectorio.
    ASSET_DIR = PROJECT_ROOT.parent / "assets" / "geospatial"

RASTER_PATH = ASSET_DIR / "escena_multibanda.tif"
PARCEL_PATH = ASSET_DIR / "parcela.geojson"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "clase_08"

assert RASTER_PATH.exists(), f"No se encontró {RASTER_PATH}"
assert PARCEL_PATH.exists(), f"No se encontró {PARCEL_PATH}"

print("FAST_MODE:", FAST_MODE)
print("GeoTIFF:", RASTER_PATH)
print("GeoJSON:", PARCEL_PATH)

In [ ]:
with rasterio.open(RASTER_PATH) as src:
    raster = src.read().astype(np.float32)
    raster_crs = src.crs
    raster_transform = src.transform
    raster_bounds = src.bounds
    raster_profile = src.profile.copy()
    band_names = src.descriptions
    pixel_size = src.res

parcelas = gpd.read_file(PARCEL_PATH)

# Validaciones esenciales antes de combinar capas.
assert raster_crs is not None, "El ráster no tiene CRS"
assert parcelas.crs is not None, "El vector no tiene CRS"
assert parcelas.crs == raster_crs, "Los CRS no coinciden"
assert raster.shape[0] == 4, "Esperábamos cuatro bandas"
assert band_names == ("blue", "green", "red", "nir")
assert not parcelas.empty

resumen = pd.DataFrame(
    {
        "propiedad": [
            "forma (bandas, alto, ancho)",
            "bandas",
            "dtype almacenado",
            "CRS ráster",
            "CRS vector",
            "resolución",
            "cantidad de parcelas",
        ],
        "valor": [
            str(raster.shape),
            ", ".join(band_names),
            raster_profile["dtype"],
            str(raster_crs),
            str(parcelas.crs),
            f"{pixel_size[0]:.0f} m × {pixel_size[1]:.0f} m",
            len(parcelas),
        ],
    }
)
display(resumen)
display(parcelas.drop(columns="geometry"))

### Leer el ráster como tensor y como mapa

El arreglo tiene forma `(4, 96, 96)`:

- eje 0: cuatro bandas;
- eje 1: filas;
- eje 2: columnas.

Para una librería de Deep Learning esto se parece a un tensor
`canales × alto × ancho`. Para una herramienta GIS no alcanza la
forma: también necesita `transform`, CRS y `nodata`.

```text
raster[band, row, col] = valor medido
             │
             └── transform(row, col) ──→ coordenada X, Y
```

El **transform afín** indica dónde está el origen de la grilla, cuánto
avanza cada columna y cuánto cambia cada fila. No lo calcularemos a
mano, pero debemos conservarlo al exportar.

### Los metadatos son parte del dato

| Propiedad | Qué controla | Falla típica si se pierde |
|---|---|---|
| `count` y descripciones | Cantidad y significado de bandas | Confundir rojo con NIR |
| `dtype` y escala | Cómo interpretar valores | Tratar `6400` como reflectancia 6400 |
| `crs` | Significado de X/Y | Capa en un lugar incorrecto |
| `transform` | Ubicación y tamaño de cada píxel | Imagen desplazada o deformada |
| `nodata` | Píxeles inválidos | Promedios contaminados |
| `bounds` | Extensión cubierta | Combinar capas que no se intersectan |

El GeoTIFF guarda reflectancias como enteros escalados. Dividiremos
por `10 000` para volver aproximadamente al rango `0..1`. Guardar
enteros reduce espacio, pero exige conocer el factor de escala.

El GeoJSON se carga como un `GeoDataFrame`: una tabla en la que una
columna especial, `geometry`, contiene objetos espaciales y el resto
conserva atributos de negocio.

In [ ]:
reflectance = raster / 10_000
blue, green, red, nir = reflectance

# RGB necesita el orden rojo-verde-azul en el último eje.
rgb = np.moveaxis(reflectance[[2, 1, 0]], 0, -1)
rgb_display = np.clip(rgb / 0.70, 0, 1)
extent = plotting_extent(raster[0], transform=raster_transform)

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(rgb_display, extent=extent)
parcelas.boundary.plot(ax=ax, color="yellow", linewidth=2)
ax.set(
    title="Composición RGB y límite de la parcela",
    xlabel="Este (m)",
    ylabel="Norte (m)",
)
plt.show()

### Preguntas para observar

1. ¿El polígono cae dentro de la imagen?
2. ¿Qué información aporta la parcela que no aporta el ráster?
3. ¿Cambiaría el resultado si las coordenadas estuvieran en grados?
4. ¿Por qué `raster.shape` no alcanza para superponer las capas?

La superposición correcta es evidencia visual útil, pero no reemplaza
la validación explícita de CRS que hicimos antes.

### Composición RGB

La visualización toma las bandas en orden rojo, verde y azul y las
mueve al último eje porque Matplotlib espera
`(alto, ancho, canales)`. La división y el recorte usados para mostrar
la imagen afectan solo su apariencia; los índices se calculan sobre
las reflectancias originales.

Una **composición falsa color** podría asignar NIR a un canal visible
para destacar vegetación. Cambiar colores de visualización no crea
información: hace observable una banda que nuestros ojos no ven.

---
## 6. NDVI, NDWI y máscaras

Un **índice espectral** combina bandas para resaltar un contraste y
reducir la cantidad de variables. En vez de analizar rojo y NIR por
separado, NDVI produce un valor por píxel.

\[
NDVI = \frac{NIR - Rojo}{NIR + Rojo}
\]

\[
NDWI = \frac{Verde - NIR}{Verde + NIR}
\]

La resta mide contraste. La división por la suma normaliza el
resultado y permite comparar mejor píxeles con distinta iluminación.

| Patrón aproximado | NDVI esperado | Explicación intuitiva |
|---|---:|---|
| NIR mucho mayor que rojo | Positivo alto | Compatible con vegetación activa |
| NIR y rojo similares | Cercano a cero | Suelo u otra cobertura |
| Rojo mayor que NIR | Negativo | Compatible con agua, sombra u otras superficies |

Esta variante de NDWI contrasta verde con NIR y suele resaltar agua
superficial. Ambos índices quedan teóricamente entre −1 y 1 porque
son diferencias normalizadas.

### Qué un índice puede y no puede decir

Un índice puede:

- resumir un patrón espectral;
- ayudar a comparar zonas o fechas compatibles;
- convertirse en feature de un modelo;
- servir como regla inicial fácil de auditar.

Un índice no puede, por sí solo:

- confirmar una enfermedad o causa;
- reemplazar observaciones de campo;
- usar el mismo umbral en cualquier sensor, estación o cultivo;
- corregir nubes, sombras o bandas mal alineadas.

> Un índice **no es un diagnóstico**. Umbrales e interpretación
> dependen del sensor, fecha, atmósfera, suelo, cultivo y objetivo.
> También existen distintas fórmulas llamadas NDWI.

Antes de comparar índices entre fechas, se deben revisar sensor,
calibración, cobertura de nubes, resolución, alineación y etapa
estacional. La fórmula correcta aplicada a datos incompatibles sigue
produciendo una comparación incorrecta.

In [ ]:
def normalized_difference(a, b):
    denominator = a + b
    return np.divide(
        a - b,
        denominator,
        out=np.zeros_like(a, dtype=np.float32),
        where=np.abs(denominator) > 1e-8,
    )


ndvi = normalized_difference(nir, red)
ndwi = normalized_difference(green, nir)

# Invariantes que evitan continuar con un resultado corrupto.
assert ndvi.shape == raster.shape[1:]
assert ndwi.shape == raster.shape[1:]
assert np.isfinite(ndvi).all() and np.isfinite(ndwi).all()
assert -1 <= float(ndvi.min()) <= float(ndvi.max()) <= 1
assert -1 <= float(ndwi.min()) <= float(ndwi.max()) <= 1

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for ax, image, title, cmap in [
    (axes[0], ndvi, "NDVI: vegetación", "RdYlGn"),
    (axes[1], ndwi, "NDWI: agua superficial", "BrBG"),
]:
    shown = ax.imshow(
        image,
        extent=extent,
        vmin=-1,
        vmax=1,
        cmap=cmap,
    )
    parcelas.boundary.plot(ax=ax, color="black", linewidth=1.5)
    ax.set_title(title)
    fig.colorbar(shown, ax=ax, shrink=0.8)
plt.show()

### De polígono a máscara

Para calcular estadísticas dentro de la parcela convertiremos su
geometría a una matriz booleana con la misma forma que el ráster:

- `True`: el píxel pertenece a la parcela;
- `False`: el píxel queda fuera.

Este paso se llama **rasterización**. La geometría vectorial se
proyecta sobre la grilla existente; no genera una grilla arbitraria.
Usamos exactamente `out_shape` y `transform` del NDVI para que cada
valor booleano corresponda al mismo píxel.

```text
polígono con coordenadas
          │
          ├── CRS compatible
          ├── transform del ráster
          └── shape del ráster
                   ↓
         máscara True / False
```

Una red de segmentación también produce máscaras. Aquí no entrenamos
una U-Net, Mask R-CNN ni SAM: practicamos la misma idea de integración
usando una geometría conocida y un umbral sencillo.

Las estadísticas calculadas dentro de una geometría se llaman
**estadísticas zonales**. Promedio, mínimo y máximo resumen la zona,
pero pueden ocultar heterogeneidad: dos parcelas con igual promedio
podrían tener distribuciones muy distintas.

In [ ]:
geometries = [geometry.__geo_interface__ for geometry in parcelas.geometry]
parcel_mask = geometry_mask(
    geometries,
    out_shape=ndvi.shape,
    transform=raster_transform,
    invert=True,
)

assert parcel_mask.shape == ndvi.shape
assert parcel_mask.any(), "La parcela no intersecta el ráster"

ndvi_in_parcel = np.where(parcel_mask, ndvi, np.nan)
NDVI_THRESHOLD = 0.45
high_vigor_mask = (ndvi >= NDVI_THRESHOLD) & parcel_mask

stats = pd.Series(
    {
        "píxeles dentro de la parcela": int(parcel_mask.sum()),
        "NDVI mínimo": float(np.nanmin(ndvi_in_parcel)),
        "NDVI promedio": float(np.nanmean(ndvi_in_parcel)),
        "NDVI máximo": float(np.nanmax(ndvi_in_parcel)),
        "porcentaje sobre umbral": (
            100 * high_vigor_mask.sum() / parcel_mask.sum()
        ),
    },
    name="valor",
)
display(stats.to_frame())

overlay = np.where(parcel_mask, high_vigor_mask.astype(float), np.nan)
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(rgb_display, extent=extent)
ax.imshow(
    overlay,
    extent=extent,
    cmap=ListedColormap(["#f59e0b", "#16a34a"]),
    alpha=0.48,
    vmin=0,
    vmax=1,
)
parcelas.boundary.plot(ax=ax, color="white", linewidth=2)
ax.set_title(
    f"Máscara dentro de parcela: NDVI ≥ {NDVI_THRESHOLD:.2f}"
)
ax.set_xlabel("Este (m)")
ax.set_ylabel("Norte (m)")
plt.show()

**Lectura de la superposición**

- verde: píxeles dentro de la parcela que superan el umbral;
- naranja: píxeles dentro de la parcela que no lo superan;
- sin color: píxeles fuera de la parcela.

La máscara integra tres piezas: una imagen, una regla o modelo, y
una geometría de negocio.

### Regla, modelo y decisión

Conviene separar tres niveles:

1. **medición:** reflectancias e índice calculado;
2. **criterio técnico:** umbral o predicción del modelo;
3. **decisión de negocio:** inspeccionar, alertar o no actuar.

El código convierte `NDVI ≥ 0.45` en una máscara, pero ese valor es
didáctico. En producción se elegiría con datos representativos,
conocimiento del dominio y una evaluación del costo de falsos
positivos y falsos negativos.

---
## 7. Exportar el NDVI como GeoTIFF

Hasta ahora `ndvi` es un arreglo en memoria. Si guardáramos solamente
sus números en un archivo genérico, otro proceso no sabría dónde
ubicarlo. Exportar un producto geoespacial significa conservar el
**contrato espacial**:

- el CRS;
- el transform que ubica cada píxel;
- ancho y alto;
- un valor `nodata`;
- tipo de dato, cantidad y descripción de bandas;
- metadatos que identifiquen origen, fórmula y versión.

```text
arreglo NDVI
    +
profile espacial del origen
    +
metadatos de linaje
    ↓
GeoTIFF derivado reutilizable
```

Usaremos `float32` porque el índice contiene decimales. El valor
`-9999` representa ausencia de información y queda fuera del rango
válido del índice. Al leer con `masked=True`, Rasterio puede
excluirlo de cálculos.

Reabrir el resultado y validar CRS, transform, forma y rango es una
prueba de contrato. Un archivo que pudo escribirse no necesariamente
es un producto correcto.

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
NDVI_OUTPUT = ARTIFACT_DIR / "ndvi_clase_08.tif"
NODATA = -9999.0

ndvi_to_write = np.where(
    np.isfinite(ndvi),
    ndvi,
    NODATA,
).astype("float32")

output_profile = raster_profile.copy()
output_profile.update(
    count=1,
    dtype="float32",
    nodata=NODATA,
    compress="deflate",
)

with rasterio.open(NDVI_OUTPUT, "w", **output_profile) as dst:
    dst.write(ndvi_to_write, 1)
    dst.set_band_description(1, "NDVI")
    dst.update_tags(
        derived_from=RASTER_PATH.name,
        formula="(nir-red)/(nir+red)",
        educational_use="true",
    )

# Reabrimos el archivo: escribir sin verificar no alcanza.
with rasterio.open(NDVI_OUTPUT) as check:
    exported_ndvi = check.read(1, masked=True)
    assert check.crs == raster_crs
    assert check.transform == raster_transform
    assert check.shape == ndvi.shape
    assert check.count == 1
    assert float(exported_ndvi.min()) >= -1
    assert float(exported_ndvi.max()) <= 1

print("Resultado exportado:", NDVI_OUTPUT)
print("CRS conservado:", raster_crs)
print("Forma conservada:", exported_ndvi.shape)

---
## 8. Actividad — elegir un umbral y justificarlo

Modificá únicamente `UMBRAL_ACTIVIDAD`. Probá al menos dos valores,
por ejemplo `0.35` y `0.60`.

No buscamos “el umbral correcto” con estos datos sintéticos. Buscamos
observar cómo una decisión técnica cambia el área reportada y
distinguir sensibilidad del algoritmo de significado agronómico.

### Hipótesis antes de ejecutar

Si aumentamos el umbral:

- menos píxeles deberían cumplir la condición;
- el contorno debería concentrarse en valores más altos;
- no necesariamente obtenemos un resultado “más verdadero”.

Anotá primero qué esperás que ocurra. Luego ejecutá y contrastá la
predicción con el resultado.

In [ ]:
# TODO: probá dos valores entre -1 y 1.
UMBRAL_ACTIVIDAD = 0.55

assert -1 <= UMBRAL_ACTIVIDAD <= 1
selected = (ndvi >= UMBRAL_ACTIVIDAD) & parcel_mask
selected_percentage = 100 * selected.sum() / parcel_mask.sum()

print(f"Umbral: {UMBRAL_ACTIVIDAD:.2f}")
print(f"Píxeles seleccionados: {selected.sum():,}")
print(f"Porcentaje de la parcela: {selected_percentage:.1f}%")

fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(
    np.where(parcel_mask, ndvi, np.nan),
    extent=extent,
    vmin=-1,
    vmax=1,
    cmap="RdYlGn",
)
ax.contour(
    selected.astype(int),
    levels=[0.5],
    extent=extent,
    colors="cyan",
    linewidths=1.5,
)
parcelas.boundary.plot(ax=ax, color="black", linewidth=2)
ax.set_title(
    f"Contorno de selección con umbral {UMBRAL_ACTIVIDAD:.2f}"
)
plt.show()

### Entregable de la actividad

Escribí cuatro conclusiones breves:

1. ¿Qué dos umbrales probaste y cuánto cambió el porcentaje?
2. ¿Qué información real necesitarías para elegir un umbral útil?
3. ¿Qué error de CRS, resolución o fecha podría invalidar el análisis?
4. ¿Qué costo tendría en el caso de uso seleccionar área de más o de menos?

**Criterios de aceptación**

- el valor está entre −1 y 1;
- el código produce una máscara con la forma del ráster;
- la interpretación distingue una medición de una decisión;
- el porcentaje no se confunde con accuracy de un modelo;
- no se presenta NDVI como diagnóstico universal.

---
## Del notebook a un pipeline de producción

El notebook procesa un archivo completo porque es pequeño. Una
escena real puede ocupar gigabytes y cubrir muchas fechas. El
sistema suele trabajar por **tiles** o teselas: bloques
rectangulares que permiten leer y procesar una parte por vez. También
debe registrar el linaje de cada producto y ejecutar controles entre
etapas.

```text
fuente
satélite / dron / catálogo / sensor
             │
             ▼
ingestión y catálogo de metadatos
sensor, fecha, nubes, CRS, licencia
             │
             ▼
preparación
corrección, ortomosaico, alineación, nodata, tiles
             │
             ▼
análisis
bandas ──→ índices ──→ features/modelo ──→ máscara
             │
             ▼
integración espacial
cruce con parcelas + estadísticas + reglas de negocio
             │
             ▼
producto
GeoTIFF / GeoJSON / base espacial / API / tablero
             │
             ▼
validación y monitoreo
campo, métricas, deriva, alertas, versionado
```

### Controles mínimos por etapa

| Riesgo | Ejemplo | Control |
|---|---|---|
| Datos equivocados | Banda roja y NIR intercambiadas | Nombres, catálogo y pruebas de rango |
| Desalineación | Máscara desplazada algunos píxeles | Comparar CRS, transform, bounds y puntos de control |
| Datos inválidos | Nubes incluidas como suelo | Máscara de calidad y tratamiento de `nodata` |
| Fuga espacial | Train y test comparten la misma parcela | Split por región/parcela/fecha |
| Cambio de dominio | Nuevo sensor o estación | Monitoreo por sensor, zona y período |
| Salida sin linaje | No se sabe qué escena generó el índice | Tags, IDs, fecha, versión de código/modelo |
| Decisión no validada | Alerta basada en umbral arbitrario | Datos de campo y revisión experta |

### Qué significa monitorear

No alcanza con medir latencia de una API. En un pipeline geoespacial
también interesa:

- porcentaje de cobertura válida y nubes;
- distribución de bandas e índices por zona y fecha;
- cantidad de geometrías inválidas o fuera de extensión;
- calidad del modelo por región, sensor y estación;
- tiempo entre adquisición, procesamiento y entrega;
- trazabilidad desde una alerta hasta imagen, modelo y configuración.

La validación de campo proporciona la referencia para saber si un
patrón espectral corresponde al fenómeno de interés. Sin ella
podemos detectar cambios en números, pero no asegurar su causa.

### Preguntas de diseño antes de construir

1. ¿Cuál es la unidad de decisión: píxel, objeto, parcela o región?
2. ¿Qué resolución y frecuencia necesita esa decisión?
3. ¿Qué errores son más costosos?
4. ¿Cómo obtendremos verdad de campo?
5. ¿Qué debe ocurrir si falta una banda o el área está nublada?
6. ¿Quién consumirá la salida y en qué formato?

En un sistema real también hay que proteger información sensible
sobre propiedades, respetar licencias, limitar accesos y definir
políticas de retención.

Esta clase recorrió la forma del pipeline. El Track Imagen
profundizará adquisición, reproyección, ortomosaicos, entrenamiento
de detectores y segmentadores, mAP/IoU sobre datasets reales,
procesamiento satelital y despliegue.

---

## Síntesis de la clase

- Un ráster organiza mediciones en píxeles; un vector representa geometrías.
- CRS, transform, extensión y resolución permiten ubicar y combinar capas.
- Las bandas multiespectrales permiten calcular índices como NDVI y NDWI.
- Una geometría puede rasterizarse como máscara para resumir una parcela.
- Clasificación, detección y segmentación producen salidas distintas que deben recuperar su contexto espacial.
- Un resultado geoespacial exportado debe conservar sus metadatos espaciales.

## Comprobación conceptual

Antes de continuar, intentá responder sin mirar el notebook:

1. ¿Cuál era el problema central de la clase?
2. ¿Qué entrada recibió el sistema y qué salida produjo?
3. ¿Qué decisión humana siguió siendo necesaria?
4. ¿Qué limitación observaste en el experimento?

Si podés explicarlo con tus propias palabras y justificarlo con un resultado visible, alcanzaste el objetivo introductorio.

## Puente con la próxima clase

Esta es la última clase de nivelación. El próximo paso es elegir el track especializado y profundizar los componentes que aquí vimos conectados.

## Conexión con los tracks

En el **Track Imagen**, estas bases se reutilizan para imágenes de dron y satélite, detección, segmentación, índices, datos agrícolas y pipelines orientados a producción.

La implementación profunda, el trabajo con datasets reales y las decisiones de producción se desarrollarán en los módulos especializados.